### Problema 2: Análisis de Vulnerabilidades con Web Scraping

In [ ]:
!pip install sentence-transformers pandas scikit-learn

In [ ]:
import requests
import re
import pandas as pd
from bs4 import BeautifulSoup
from datetime import datetime, date
from dateutil.relativedelta import relativedelta

In [ ]:
# ============================================================
# CONFIGURACIÓN
# ============================================================

BASE_URL = "https://lists.debian.org/debian-security-announce/2026/"
FECHA_FIN = date.today()
FECHA_INICIO = FECHA_FIN - relativedelta(months=3)

print("Fecha inicio:", FECHA_INICIO)
print("Fecha fin:", FECHA_FIN)


# ============================================================
# FUNCIÓN PARA EXTRAER LOS DATOS DE UN AVISO
# ============================================================

def extraer_datos(texto):

    # ========================================================
    # FECHA
    # ========================================================

    fecha_match = re.search(
        r"Debian Security Advisory DSA-\d+-\d+.*?\n.*?\n.*?\n([A-Z][a-z]+ \d{1,2}, \d{4})",
        texto,
        re.DOTALL
    )

    fecha = None

    if fecha_match:
        fecha = datetime.strptime(
            fecha_match.group(1),
            "%B %d, %Y"
        ).date()


    # ========================================================
    # PACKAGE
    # ========================================================

    package_match = re.search(
        r"Package\s*:\s*(.+)",
        texto
    )

    package = (
        package_match.group(1).strip()
        if package_match
        else None
    )


    # ========================================================
    # CVE
    # ========================================================

    """cve_match = re.search(
        r"CVE ID\s*:\s*(.+)",
        texto
    )

    cves = (
        cve_match.group(1).strip()
        if cve_match
        else None
    )"""
    import re

    # 1. Definimos la nueva función arriba en tu script
    def extraer_todos_los_cves(texto_bloque):
        if not texto_bloque:
            return "not yet available"
        # Busca todos los patrones CVE-AAAA-NNNN sin importar los saltos de línea
        cves_encontrados = re.findall(r"CVE-\d{4}-\d+", texto_bloque)
        # Quitamos duplicados por si acaso
        cves_unicos = list(dict.fromkeys(cves_encontrados))
        
        if cves_unicos:
            return " ".join(cves_unicos)
        return "not yet available"


    # =====================================================================
    # 2. ASÍ LO APLICÁS EN TU BUCLE DE SCRAPING:
    # =====================================================================

    # Primero buscamos dónde empieza el bloque de CVEs
    cve_match = re.search(r"CVE ID\s*:\s*([\s\S]+?)(?=\n[A-Z][a-z]+|\r?\n-{3,}|$)", texto)

    if cve_match:
        # Capturamos todo el bloque de texto (incluyendo sus saltos de línea)
        bloque_cves_sucio = cve_match.group(1)
        
        # ¡AQUÍ MANDÁS LA FUNCIÓN NUEVA!: Limpia y extrae absolutamente todos los CVEs de ese bloque
        cves = extraer_todos_los_cves(bloque_cves_sucio)
    else:
        cves = "not yet available"



    # ========================================================
    # DESCRIPCIÓN
    # ========================================================

    descripcion = None

    # Buscamos dónde termina el bloque de metadatos.
    # CVE ID es obligatorio en la mayoría de los avisos,
    # pero algunos pueden decir "not yet available".

    inicio_match = re.search(
        r"CVE ID\s*:\s*.*?\n",
        texto
    )

    if inicio_match:

        inicio = inicio_match.end()

        # Sacamos el texto posterior
        texto_posterior = texto[inicio:]

        # Eliminamos posibles campos adicionales de metadatos
        texto_posterior = re.sub(
            r"^\s*Debian Bug\s*:.*?\n",
            "",
            texto_posterior,
            flags=re.MULTILINE
        )

        # Eliminamos espacios y saltos iniciales
        texto_posterior = texto_posterior.strip()


        # ====================================================
        # BUSCAR DÓNDE TERMINA LA DESCRIPCIÓN
        # ====================================================

        patrones_fin = [
            r"\nFor the stable distribution",
            r"\nFor the oldstable distribution",
            r"\nFor the oldoldstable distribution",
            r"\nWe recommend that you upgrade",
            r"\nFor the detailed security status",
        ]

        posiciones = []

        for patron in patrones_fin:

            match_fin = re.search(
                patron,
                texto_posterior
            )

            if match_fin:
                posiciones.append(match_fin.start())


        if posiciones:

            fin = sorted(posiciones)[0]

            descripcion = texto_posterior[:fin].strip()

        else:

            descripcion = texto_posterior.strip()


        # Limpiamos saltos de línea
        descripcion = " ".join(
            descripcion.split()
        )


        return {
            "fecha": fecha,
            "package": package,
            "cves": cves,
            "descripcion": descripcion
        }

    # -------------------------
    # CVE
    # -------------------------

    cve_match = re.search(
        r"CVE ID\s*:\s*(.+)",
        texto
    )

    cves = (
        cve_match.group(1).strip()
        if cve_match
        else None
    )


    # -------------------------
    # Descripción
    # -------------------------

    descripcion_match = re.search(
        r"Debian Bug\s*:.*?\n\n(.*?)\n\nFor the stable distribution",
        texto,
        re.DOTALL
    )

    descripcion = None

    if descripcion_match:
        descripcion = " ".join(
            descripcion_match.group(1).split()
        )


    return {
        "fecha": fecha,
        "package": package,
        "cves": cves,
        "descripcion": descripcion
    }



Fecha inicio: 2026-06-17
Fecha fin: 2026-09-17

Cantidad de avisos encontrados: 416

[1/416] Procesando msg00415.html
  Fecha: 2026-09-16
  Package: thunderbird
  CVE: CVE-2026-92005 CVE-2026-92006 CVE-2026-92007 CVE-2026-92008

[2/416] Procesando msg00414.html
  Fecha: 2026-09-16
  Package: mkvtoolnix
  CVE: CVE-2026-90783

[3/416] Procesando msg00413.html
  Fecha: 2026-09-16
  Package: firefox-esr
  CVE: CVE-2026-92005 CVE-2026-92006 CVE-2026-92007 CVE-2026-92008

[4/416] Procesando msg00412.html
  Fecha: 2026-09-16
  Package: tor
  CVE: not yet available

[5/416] Procesando msg00411.html
  Fecha: 2026-09-16
  Package: nginx
  CVE: None

[6/416] Procesando msg00410.html
  Fecha: 2026-09-15
  Package: cjose
  CVE: CVE-2026-53938 CVE-2026-53939

[7/416] Procesando msg00409.html
  Fecha: 2026-09-14
  Package: network-manager-l2tp
  CVE: CVE-2026-19624 CVE-2026-75131 CVE-2026-75883

[8/416] Procesando msg00408.html
  Fecha: 2026-09-12
  Package: xorg-server
  CVE: CVE-2026-55999 CVE-2026

In [ ]:
# ============================================================
# OBTENER LINKS DEL ARCHIVO 2026
# ============================================================

response = requests.get(BASE_URL)

response.raise_for_status()

soup = BeautifulSoup(response.text, "html.parser")


links = []

for a in soup.find_all("a"):

    href = a.get("href")

    if href and re.match(r"msg\d+\.html$", href):
        links.append(href)


print("\nCantidad de avisos encontrados:", len(links))


# ============================================================
# ORDENAR LOS AVISOS POR NÚMERO
# ============================================================

links.sort(
    key=lambda x: int(
        re.search(r"msg(\d+)\.html", x).group(1)
    ),
    reverse=True
)

In [ ]:
# ============================================================
# RECORRER LOS AVISOS
# ============================================================

resultados = []


for i, link in enumerate(links):

    url = BASE_URL + link

    print(f"\n[{i + 1}/{len(links)}] Procesando {link}")

    try:

        response = requests.get(url)

        response.raise_for_status()

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        texto = soup.get_text("\n", strip=True)

        datos = extraer_datos(texto)


        # Si no pudimos obtener la fecha,
        # continuamos con el siguiente aviso

        if datos["fecha"] is None:

            print("  No se pudo obtener la fecha")
            continue


        print("  Fecha:", datos["fecha"])
        print("  Package:", datos["package"])
        print("  CVE:", datos["cves"])


        # ====================================================
        # FILTRO DE LOS ÚLTIMOS 3 MESES
        # ====================================================

        if datos["fecha"] > FECHA_FIN:

            continue


        if datos["fecha"] < FECHA_INICIO:

            print("  Aviso anterior a los últimos 3 meses.")
            print("  Deteniendo scraping.")

            break


        # ====================================================
        # GUARDAR
        # ====================================================

        datos["dsa"] = re.search(
            r"DSA-\d+-\d+",
            texto
        ).group(0)

        datos["url"] = url

        resultados.append(datos)


    except requests.RequestException as e:

        print("  Error:", e)


In [ ]:
# ============================================================
# DATAFRAME
# ============================================================

df = pd.DataFrame(resultados)


# Ordenamos por fecha

df = df.sort_values(
    "fecha",
    ascending=False
).reset_index(drop=True)


print("\n===================================")
print("RESULTADO")
print("===================================")

print("Cantidad de avisos:", len(df))

print(df.head())


# ============================================================
# GUARDAR CSV
# ============================================================

df.to_csv(
    "debian_security_advisories_ultimos_3_meses.csv",
    index=False
)

print(
    "\nArchivo guardado como:",
    "debian_security_advisories_ultimos_3_meses.csv"
)